# Learning-Augmented Treap 实验分析

本 notebook 读取 `src/LA_BST_CSV.cpp` 输出的 CSV（`data/e1_*.csv` … `e7_*.csv`），对 E1–E7 七个实验做表格汇总与交互式可视化。

运行前请先按 README 的说明编译并运行 C++ 程序，生成 `data/*.csv`；也可先执行 `python notebooks/plot_all.py` 生成 PDF 图表，再运行本 notebook 查看交互式结果。

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# 定位项目根目录：若当前目录包含 data/ 则认为是根目录，否则回退到上一级（notebooks/ 目录）
cwd = os.getcwd()
ROOT = cwd if os.path.isdir(os.path.join(cwd, 'data')) else os.path.dirname(cwd)
DATA = os.path.join(ROOT, 'data')
print('数据目录:', DATA)

def load(name):
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        print('[warn] 缺少', path, '——请先运行 C++ 程序生成 data/*.csv')
        return None
    return pd.read_csv(path)

e1 = load('e1_basic.csv')
e1_seed = load('e1_per_seed.csv')
e2 = load('e2_drift.csv')
e3 = load('e3_singularity.csv')
e4 = load('e4_static_optimal.csv')
e5 = load('e5_kl_noise.csv')
e6 = load('e6_adaptive_perturb.csv')
e7 = load('e7_bucketing.csv')

In [ ]:
# 设计系统参数（与 plot_all.py 一致：dataviz 参考调色板，浅色模式）
CAT = {
    'Random': '#2a78d6',
    'Chen': '#eb3434',
    'f1': '#d2c400',
    'f2': '#00b812bd',
    'f3': '#eb38e5',
    'f4': '#9007C6',
}
METHOD_ORDER = ['Random', 'Chen', 'f1', 'f2', 'f3', 'f4']

plt.rcParams.update({
    'figure.facecolor': '#fcfcfb',
    'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7',
    'axes.labelcolor': '#0b0b0b',
    'axes.titlecolor': '#0b0b0b',
    'xtick.color': '#52514e',
    'ytick.color': '#52514e',
    'grid.color': '#e1e0d9',
    'grid.linestyle': '--',
    'grid.linewidth': 0.6,
    'axes.grid': True,
    'axes.axisbelow': True,
    'font.size': 11,
    'font.sans-serif': ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans'],
    'axes.unicode_minus': False,
    'legend.frameon': False,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

def mcolor(m):
    return CAT.get(m, '#888888')

## E1 常规分布基础性能

在 Zipf / Adversarial / Uniform 分布上，对比随机基线（Random）、Chen 双对数与 `f1`/`f3`/`f4` 的总比较次数、平均深度与最大深度。

In [ ]:
if e1 is not None:
    tab = e1.pivot_table(index=['distribution', 'param', 'n'], columns='method', values='mean_comparisons')
    display(tab.round(0))

    sel = e1[(e1['distribution'] == 'Zipf') & (e1['param'] == 1.0)]
    methods = [m for m in METHOD_ORDER if m in set(sel['method'])]
    ns = sorted(sel['n'].unique())
    n_colors = {ns[0]: '#86b6ef', ns[1]: '#1c5cab'}

    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(methods))
    width = 0.38
    for i, n in enumerate(ns):
        sub = sel[sel['n'] == n].set_index('method')
        vals = [sub.loc[m, 'mean_comparisons'] for m in methods]
        errs = [sub.loc[m, 'std_comparisons'] for m in methods]
        bx = x + (i - 0.5) * width
        ax.bar(bx, vals, width, color=n_colors[n], label=f'n = {n}', zorder=3)
        ax.errorbar(bx, vals, yerr=errs, fmt='none', ecolor='#52514e', elinewidth=0.8, capsize=3, zorder=4)
    ax.set_xticks(x)
    ax.set_xticklabels(methods)
    ax.set_xlabel('方法')
    ax.set_ylabel('总比较次数（m = 10⁵）')
    ax.set_title('E1  Zipf α = 1 下各方法总比较次数')
    ax.legend(title='规模')
    ax.margins(x=0.05)
    fig.tight_layout()
    plt.show()

### 附：E1 逐种子分布

`e1_per_seed.csv` 记录了 30 个随机种子下每个方法的逐次平均比较次数，用箱线图观察跨种子稳定性。

In [ ]:
if e1_seed is not None:
    sel = e1_seed[(e1_seed['distribution'] == 'Zipf') & (e1_seed['param'] == 1.0) & (e1_seed['n'] == 5000)]
    methods = [m for m in METHOD_ORDER if m in set(sel['method'])]
    data = [sel[sel['method'] == m]['avg_comparisons'].values for m in methods]

    fig, ax = plt.subplots(figsize=(8, 5))
    bp = ax.boxplot(data, labels=methods, patch_artist=True, showfliers=False)
    for patch, m in zip(bp['boxes'], methods):
        patch.set_facecolor(mcolor(m))
        patch.set_alpha(0.55)
    ax.set_xlabel('方法')
    ax.set_ylabel('逐种子平均比较次数（m = 10⁵）')
    ax.set_title('E1  逐种子比较次数分布（Zipf α=1, n=5000, 30 seeds）')
    fig.tight_layout()
    plt.show()

## E2 绝对量纲漂移鲁棒性

对完美频率施加缩放 `w → c·w` 与幂律 `w → w^β` 两种严格单调变换，观察各方法的查询代价与 Kendall τ（τ = 1 表示拓扑完全不变）。

In [ ]:
if e2 is not None:
    tab = e2.pivot_table(index=['transform', 'param'], columns='method', values='mean_kendall_tau')
    display(tab.round(3))

    df = e2.copy()
    df['label'] = df['transform'] + '=' + df['param'].astype(str)
    labels = sorted(df['label'].unique())
    methods = [m for m in METHOD_ORDER if m in set(df['method'])]
    x = np.arange(len(labels))
    w = 0.18

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for i, m in enumerate(methods):
        sub = df[df['method'] == m].set_index('label')
        vals = [sub.loc[l, 'mean_comparisons'] for l in labels]
        errs = [sub.loc[l, 'std_comparisons'] for l in labels]
        bx = x + (i - (len(methods) - 1) / 2) * w
        axes[0].bar(bx, vals, w, color=mcolor(m), label=m, zorder=3)
        axes[0].errorbar(bx, vals, yerr=errs, fmt='none', ecolor='#52514e', elinewidth=0.7, capsize=2, zorder=4)
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
    axes[0].set_xlabel('量纲变换'); axes[0].set_ylabel('总比较次数')
    axes[0].set_title('比较次数')

    for i, m in enumerate(methods):
        sub = df[df['method'] == m].set_index('label')
        vals = [sub.loc[l, 'mean_kendall_tau'] for l in labels]
        bx = x + (i - (len(methods) - 1) / 2) * w
        axes[1].bar(bx, vals, w, color=mcolor(m), label=m, zorder=3)
    axes[1].axhline(1.0, color='#c3c2b7', lw=0.8, ls=':')
    axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
    axes[1].set_ylim(0, 1.05)
    axes[1].set_xlabel('量纲变换'); axes[1].set_ylabel('Kendall τ')
    axes[1].set_title('拓扑不变性（τ = 1 表示拓扑不变）')

    axes[0].legend(ncol=2)
    fig.suptitle('E2  绝对量纲漂移的鲁棒性', fontsize=13)
    fig.tight_layout()
    plt.show()

## E3 右端奇点与数值溢出

把预测权重 `w` 推向 1，检查各映射函数的最大 priority 是否溢出（inf/nan）。

In [ ]:
if e3 is not None:
    display(e3)

    df = e3.copy()
    df['max_priority'] = df['max_priority'].replace([np.inf, -np.inf], np.nan)
    ws = sorted(df['w'].unique())
    wlabels = [f'{w:g}' for w in ws]
    methods = [m for m in METHOD_ORDER if m in set(df['method'])]
    x = np.arange(len(ws))

    fig, ax = plt.subplots(figsize=(8, 5))
    for m in methods:
        sub = df[df['method'] == m].set_index('w')
        vals = [sub.loc[w, 'max_priority'] for w in ws]
        ax.plot(x, vals, marker='o', markersize=5, lw=1.6, color=mcolor(m), label=m, zorder=3)
    ax.annotate('Chen → ∞（溢出）', xy=(len(ws) - 1, 32), xytext=(len(ws) - 2.2, 30),
                color=CAT['Chen'], arrowprops=dict(arrowstyle='->', color=CAT['Chen']))
    ax.axhline(0, color='#c3c2b7', lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(wlabels)
    ax.set_xlabel('预测权重 w（右端奇点）')
    ax.set_ylabel('最大 priority')
    ax.set_ylim(-6, 34)
    ax.set_title('E3  右端奇点压力测试：Chen 发散，其余有界')
    ax.legend()
    fig.tight_layout()
    plt.show()

## E4 理论静态最优代价

用静态最优比 `ρ = 总比较次数 / 香农熵下界` 量化各方法相对绝对频率静态最优的偏离。

In [ ]:
if e4 is not None:
    tab = e4.pivot_table(index=['distribution', 'param', 'n'], columns='method', values='mean_rho')
    display(tab.round(3))

    df = e4[e4['n'] == 5000].copy()
    def dist_label(r):
        if r['distribution'] == 'Zipf':
            return r['distribution'] + ' α=' + str(r['param'])
        return r['distribution']
    df['dist'] = df.apply(dist_label, axis=1)
    dists = sorted(df['dist'].unique())
    methods = [m for m in METHOD_ORDER if m in set(df['method'])]
    x = np.arange(len(dists))
    w = 0.14

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, m in enumerate(methods):
        sub = df[df['method'] == m].set_index('dist')
        vals = [sub.loc[d, 'mean_rho'] for d in dists]
        errs = [sub.loc[d, 'std_rho'] for d in dists]
        bx = x + (i - (len(methods) - 1) / 2) * w
        ax.bar(bx, vals, w, color=mcolor(m), label=m, zorder=3)
        ax.errorbar(bx, vals, yerr=errs, fmt='none', ecolor='#52514e', elinewidth=0.7, capsize=2, zorder=4)
    ax.axhline(1.0, color='#c3c2b7', lw=0.8, ls=':')
    ax.set_xticks(x); ax.set_xticklabels(dists)
    ax.set_xlabel('分布')
    ax.set_ylabel('静态最优比 ρ = 比较次数 / 香农熵下界')
    ax.set_title('E4  静态最优比（n = 5000，ρ 越接近常数越优）')
    ax.legend(ncol=3)
    fig.tight_layout()
    plt.show()

## E5 非完美预测器（KL 噪声）鲁棒性

对预测分布注入乘性对数正态噪声，二分搜索 σ 使 `D_KL(P ‖ Q)` 逼近目标值，观察各方法随噪声增大的退化。

In [ ]:
if e5 is not None:
    tab = e5.pivot_table(index='kl', columns='method', values='mean_comparisons')
    display(tab.round(0))

    df = e5.copy()
    methods = [m for m in METHOD_ORDER if m in set(df['method'])]
    fig, ax = plt.subplots(figsize=(8, 5))
    for m in methods:
        sub = df[df['method'] == m].sort_values('kl')
        ax.errorbar(sub['kl'], sub['mean_comparisons'], yerr=sub['std_comparisons'],
                    color=mcolor(m), marker='o', markersize=5, lw=1.6,
                    capsize=3, label=m, zorder=3, elinewidth=0.8, ecolor=mcolor(m))
    ax.set_xlabel('KL 散度  D_KL(P ‖ Q)')
    ax.set_ylabel('总比较次数（m = 10⁵）')
    ax.set_title('E5  预测器 KL 噪声下的鲁棒性')
    ax.set_xticks(sorted(df['kl'].unique()))
    ax.legend(ncol=2)
    fig.tight_layout()
    plt.show()

## E6 自适应扰动 vs 固定幅度扫描

扫描固定扰动幅度 `φ = 0.01 … 2.0`，并与自适应扰动（`δ_max = min(0.49, 0.49·gap)`）对比。跨桶逆序率是机制指标，查询代价是后果指标。

In [ ]:
if e6 is not None:
    display(e6)

    df = e6.copy()
    methods = ['f3', 'f4']
    fixed_order = ['0.01', '0.25', '0.49', '0.75', '0.99', '1.25', '1.75', '2.0']
    x = np.arange(len(fixed_order))
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

    for m in methods:
        sub = df[df['method'] == m].set_index('perturb')
        fixed = sub.loc[fixed_order]
        axes[0].plot(x, fixed['mean_inversion_ratio'], marker='o', markersize=5, lw=1.8,
                     color=CAT[m], label=m, zorder=3)
        axes[0].fill_between(x,
            fixed['mean_inversion_ratio'] - fixed['std_inversion_ratio'],
            fixed['mean_inversion_ratio'] + fixed['std_inversion_ratio'],
            color=CAT[m], alpha=0.15, zorder=2)
        axes[1].errorbar(x, fixed['mean_comparisons'], yerr=fixed['std_comparisons'],
                         color=CAT[m], marker='o', markersize=5, lw=1.8,
                         capsize=3, elinewidth=0.8, ecolor=CAT[m], label=m, zorder=3)

    for m in methods:
        sub = df[(df['method'] == m) & (df['perturb'] == 'adaptive')]
        if len(sub) == 0:
            continue
        axes[0].axhline(sub['mean_inversion_ratio'].iloc[0], color=CAT[m], ls='--', lw=1.1, alpha=0.7)
        axes[1].axhline(sub['mean_comparisons'].iloc[0], color=CAT[m], ls='--', lw=1.1, alpha=0.7)

    for ax in axes:
        ax.axvline(4.5, color='#c3c2b7', lw=0.9, ls=':')
        ax.set_xticks(x)
        ax.set_xticklabels(fixed_order)
        ax.set_xlabel('固定扰动幅度 φ（桶宽 = 1）')

    axes[0].set_ylabel('跨桶逆序率')
    axes[0].set_ylim(-0.005, 0.14)
    axes[0].set_title('机制指标：跨桶逆序率（φ ≥ 1 从 0 跳升）')
    axes[1].set_ylabel('总比较次数（m = 10⁵）')
    axes[1].set_title('后果指标：查询代价')

    dash = plt.Line2D([], [], color='#52514e', ls='--', lw=1.2, label='自适应（虚线，恒安全）')
    for ax in axes:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles + [dash], labels=labels + [dash.get_label()], ncol=3, fontsize=9)

    fig.suptitle('E6  自适应扰动：固定幅度跨过桶宽（φ ≥ 1）即破坏学习排序', fontsize=13)
    fig.tight_layout()
    plt.show()

## E7 分桶 + 扰动必要性消融

在对抗键序（key 0 最热、插入序 = 键序、去掉随机化）下，消融分桶与扰动对树高和查询代价的影响。

In [ ]:
if e7 is not None:
    display(e7)

    df = e7.copy()
    labels = {
        'no_bucket_no_perturb': '无分桶·无扰动',
        'bucket_no_perturb': '分桶·无扰动',
        'bucket_perturb': '分桶·固定扰动',
        'bucket_adaptive': '分桶·自适应扰动',
    }
    settings = list(df['setting'])
    x = np.arange(len(settings))
    colors = ['#d03b3b', '#d03b3b', '#0ca30c', '#0ca30c']
    metrics = [
        ('mean_height', '树高（n = 5000）'),
        ('mean_comparisons', '总比较次数（m = 2×10⁴）'),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
    for ax, (mc, ylab) in zip(axes, metrics):
        vals = df[mc].values
        ax.bar(x, vals, 0.5, bottom=1, color=colors, zorder=3)
        ax.set_yscale('log')
        ax.set_ylim(bottom=1)
        for xi, v in zip(x, vals):
            ax.annotate(f'{v:,.0f}', (xi, v), textcoords='offset points', xytext=(0, 3),
                        ha='center', fontsize=8.5, color='#52514e')
        ax.set_xticks(x)
        ax.set_xticklabels([labels[s] for s in settings])
        ax.set_ylabel(ylab)
    axes[0].set_title('树高（退化 = 链，≈ n）')
    axes[1].set_title('查询代价')
    fig.suptitle('E7  分桶 + 扰动必要性消融（对抗键序，Zipf α=1, n=5000）', fontsize=13)
    fig.tight_layout()
    plt.show()

## 总结

- E1：`f1`/`f3`/`f4` 在常规 Zipf 分布下比较次数均低于随机基线，`f3` 略优于 Chen。
- E2：基于秩的 `f3`/`f4` 在缩放与幂律变换下 Kendall τ 恒为 1，Chen 与 `f1` 发生拓扑畸变。
- E3：`w = 1` 时 Chen 溢出为 ∞，`f1`/`f3`/`f4` 有界。
- E4：Chen 与 `f1` 的静态最优比 ρ 接近 1；`f2` 在偏斜分布下 ρ 明显升高；`f3`/`f4` 以 KL 散度代价换取尺度不变性。
- E5：各方法随预测器 KL 噪声增大而退化，随机基线不依赖预测器故不受影响。
- E6：固定扰动幅度 φ ≥ 1（跨过桶宽）时跨桶逆序率从 0 跳升，查询代价随之上升；自适应扰动恒安全。
- E7：无分桶时树退化成链，分桶 + 扰动把树高与比较次数压回 O(log n) 量级。

完整理论推导与逐条缺陷分析见 `Documents and Files/Priority_Design.md`。